# Simulating the Zhang et al. (2023) flume tests

Zhang, R., Chen, Y., Lei, J., Zhou, X., Yao, P., Stive, M.J.F. (2023). *Experimental
investigation of wave attenuation by mangrove forests with submerged canopies.*
Coastal Engineering 186, 104403. doi:[10.1016/j.coastaleng.2023.104403](https://doi.org/10.1016/j.coastaleng.2023.104403)

The [reproduction notebook](02_zhang2023_reproduction.ipynb) checks that SADIE rebuilds
the paper's geometry, closures and damping factors. This notebook asks a different
question: **if we simulate the experiment with SADIE, how close are the predicted wave
heights to the heights measured in the flume?**

The only gauge measurements published are the 24 wave-height profiles in Fig. 4 (p. 7):
H(x)/H0 at gauges G4–G8 (x/B = 0, 0.25, 0.5, 0.75, 1) for nine wave conditions and
the three mangrove models. The paper says the raw data are available on request; this
notebook uses values **digitized from Fig. 4**. As in the paper, the heights are for the
vegetation alone: the bare-bed (M0) reduction has been subtracted (p. 7).

Each simulation uses only the model geometry, the wave condition and a drag closure. Three
kinds of closure are compared, and they say different things:

1. **Zhang et al.'s empirical damping model** (HSVF, Eqs. 13–15). It was fitted to all
   104 tests, including these 24, separately for each model.
2. **SADIE physics with Zhang et al.'s pooled $C_D$ closures** (Table 3). They were fitted
   to the same experiment, pooled over M1–M3. This is a consistency check.
3. **SADIE physics with $C_D$ closures from other experiments** (Table 3). These are true
   predictions: none of these data were used to fit them.

Page numbers refer to the journal pages of the paper.

In [ ]:
import csv
import warnings
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

import sadiepy as sd
from sadiepy.presets.zhang2023 import FIG4_BETA, PATCH_LENGTH, beta_hsvf

# Several runs lie outside a closure's fitted range; the warnings are recorded on each
# result. Silence the repeats to keep the notebook readable.
warnings.simplefilter("ignore", sd.ValidityWarning)  # includes ExtrapolationWarning

water = sd.Water(temperature=20.0)  # the paper does not state the water temperature
COLOR = {"M1": "#2a78d6", "M2": "#eb6834", "M3": "#1baf7a"}  # validated categorical slots
MARKER = {"M1": "o", "M2": "s", "M3": "^"}
CONFIGS = ("M1", "M2", "M3")

## 1. The measurements

`data/zhang2023_fig4_digitized.csv` holds the 120 gauge points (24 profiles × 5 gauges).
They were extracted from a 400-dpi render of Fig. 4 by marker colour; the script
`data/digitize_zhang2023_fig4.py` records the method. Three markers partly hidden by a
fitted line were read from their outline, and two hidden behind the legend or another
marker were read by eye. The `method` column flags each of them.

The axes span 0–1.2 over about 615 px, so one pixel is 0.002 in H/H0; allowing for marker
size and anti-aliasing, the digitizing uncertainty is about ±0.005.

In [ ]:
MEAS: dict[tuple[float, float, float, str], np.ndarray] = defaultdict(list)
with open("data/zhang2023_fig4_digitized.csv") as f:
    for row in csv.DictReader(f):
        key = (float(row["H_m"]), float(row["T_s"]), float(row["h_m"]), row["config"])
        MEAS[key].append(float(row["H_over_H0"]))
MEAS = {k: np.array(v) for k, v in MEAS.items()}
X = np.linspace(0.0, PATCH_LENGTH, 5)  # gauge positions G4-G8 in the patch, m
CASES = sorted(MEAS, key=lambda k: (k[2], k[1], k[0], k[3]))
WAVES = sorted({k[:3] for k in CASES}, key=lambda w: (w[2], w[1], w[0]))
print(f"{len(CASES)} profiles, {sum(len(v) for v in MEAS.values())} gauge points")

As a check on the digitizing, fit Eq. (3), H/H0 = 1/(1 + βx), to each digitized profile
by least squares and compare it with the β printed in the panel.

In [ ]:
from scipy.optimize import curve_fit


def fit_beta(h_ratio: np.ndarray) -> float:
    popt, _ = curve_fit(lambda x, b: 1.0 / (1.0 + b * x), X, h_ratio, p0=[0.1])
    return float(popt[0])


lines = [
    "| H (m) | T (s) | h (m) | model | β printed | β refit | difference |",
    "|---|---|---|---|---|---|---|",
]
for k in CASES:
    printed, refit = FIG4_BETA[k[:3]][k[3]], fit_beta(MEAS[k])
    cells = (*k, f"{printed:.3f}", f"{refit:.3f}", f"{refit - printed:+.3f}")
    lines.append("| " + " | ".join(map(str, cells)) + " |")
display(Markdown("\n".join(lines)))

Most profiles refit to within 0.005 of the printed β. Two do not: panel (a3),
H = 0.08 m, T = 1.6 s, h = 0.3 m, where the M3 refit is 0.34 against 0.29 printed and M1
is 0.039 against 0.047. In that panel the printed curve misses the plotted G5 point
(0.76 against about 0.87 on the curve), so the paper's fit cannot be an unweighted
least-squares fit to the plotted points. The low-β M1 profiles at h = 0.5 m differ by up
to 0.004, which is within the digitizing uncertainty for profiles that decay by only 3–4 %.

Below, "measured β" means the β printed in Fig. 4.

## 2. The simulation

For each profile, SADIE builds the model from the `zhang2023` preset (as-built geometry,
with the M3 canopy area following Fig. 3a; see the reproduction notebook, §1) and
propagates the incident wave through the 2 m patch with the energy balance of the
[theory](../theory.md) page (§3).

The setup follows the way Zhang et al. analysed their data, because that is how their
closures were fitted:

- **Depth-averaged geometry.** The frontal area enters Eq. (4) as the depth average
  $\bar D$ (theory Eq. 5.5), not resolved layer by layer.
- **Incident evaluation.** $C_D$ is evaluated once at the incident wave height H0, as in
  the paper's fits (theory §5.4).
- **Source-scale Reynolds numbers.** Reynolds-number closures are evaluated at the
  Reynolds number converted to the scale of the experiment they came from (theory Eq. 5.6).

Section 5 repeats the comparison with the resolved geometry and local $C_D$ (SADIE's
default) and with Zhang et al.'s inferred root convention.

In [ ]:
def simulate(case, drag, geometry="depth", evaluation="incident", root_frontal_factor=1.0):
    """Predicted H/H0 at the gauges and the predicted beta for one profile."""
    H, T, h, config = case
    wave = sd.RegularWave(H, T, h)
    veg = sd.presets.zhang2023(config, root_frontal_factor=root_frontal_factor)
    if geometry == "depth":
        veg = veg.depth_averaged(h)
    res = sd.propagate(
        wave,
        veg,
        PATCH_LENGTH,
        water=water,
        drag=drag,
        x_out=X,
        cd_evaluation=evaluation,
        scale=10.0,
    )
    return res.H / H, res.beta


def hsvf_model(case):
    """Zhang et al.'s empirical model, Eqs. (13)-(15): beta = psi * HSVF."""
    beta = beta_hsvf(case[3], sd.RegularWave(*case[:3]))
    return 1.0 / (1.0 + beta * X), beta


def paper_fit(case):
    """The paper's own fitted curve (printed beta): the best any beta model can do."""
    beta = FIG4_BETA[case[:3]][case[3]]
    return 1.0 / (1.0 + beta * X), beta

## 3. Predicted and measured wave heights

Markers are the measured gauge heights (Fig. 4). Solid lines are SADIE with Zhang et al.'s
KC closure (`zhang2023_kc`); dashed lines are Zhang et al.'s empirical HSVF model.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10, 9), sharex=True, sharey=True, layout="constrained")
xf = X / PATCH_LENGTH
for ax, wave in zip(axes.flat, WAVES, strict=True):
    for c in CONFIGS:
        case = (*wave, c)
        if case not in MEAS:
            continue
        sadie, _ = simulate(case, "zhang2023_kc")
        emp, _ = hsvf_model(case)
        ax.plot(xf, sadie, "-", color=COLOR[c], lw=2)
        ax.plot(xf, emp, "--", color=COLOR[c], lw=1.5)
        ax.scatter(
            xf,
            MEAS[case],
            s=45,
            color=COLOR[c],
            marker=MARKER[c],
            edgecolor="white",
            linewidth=1,
            zorder=3,
            label=c,
        )
    ax.set_title(f"H = {wave[0]} m, T = {wave[1]} s, h = {wave[2]} m", fontsize=9)
    ax.set_ylim(0.2, 1.08)
    ax.grid(color="0.9")
for ax in axes[-1]:
    ax.set_xlabel("x / B")
for ax in axes[:, 0]:
    ax.set_ylabel("H(x) / H0")
handles = [
    plt.Line2D([], [], color=COLOR[c], marker=MARKER[c], ls="", label=f"{c} measured")
    for c in CONFIGS
]
handles += [
    plt.Line2D([], [], color="0.3", lw=2, label="SADIE + zhang2023_kc"),
    plt.Line2D([], [], color="0.3", lw=1.5, ls="--", label="Zhang Eqs. 13–15 (HSVF)"),
]
fig.legend(handles=handles, loc="outside lower center", ncol=5, frameon=False)
plt.show()

The canopy models M2 and M3 are close, and the empirical model is closer still. M1 is
not: SADIE with the pooled closure predicts far more damping than measured, at every depth.
Section 4 puts numbers on this.

## 4. Scores for every closure

For each prediction:

- **RMSE** and **bias** of H/H0 over the 96 gauge points inside the patch (x/B > 0).
  Bias is predicted minus measured, so a negative bias means too much damping.
- **β ratio:** the geometric mean of predicted/measured β, for each model. A value of 1 is
  exact; 2 means twice the measured damping.

The first row, the paper's own fitted curves, sets the floor: it is the scatter of the
measured points about Eq. (3), plus the digitizing uncertainty.

In [ ]:
TIER = {"zhang2023_kc": 2, "zhang2023_rerv": 2, "zhang2023_ur": 2}
TIER |= {n: 3 for n in sd.core.drag.names() if n not in TIER}


def score(model) -> dict:
    err, ratios = [], defaultdict(list)
    for case in CASES:
        pred, beta = model(case)
        err.extend(pred[1:] - MEAS[case][1:])
        ratios[case[3]].append(np.log(beta / FIG4_BETA[case[:3]][case[3]]))
    err = np.array(err)
    return {
        "rmse": float(np.sqrt(np.mean(err**2))),
        "bias": float(err.mean()),
        **{c: float(np.exp(np.mean(ratios[c]))) for c in CONFIGS},
    }


def table(rows: dict[str, dict]) -> Markdown:
    out = [
        "| prediction | kind | RMSE | bias | β ratio M1 | β ratio M2 | β ratio M3 |",
        "|---|---|---|---|---|---|---|",
    ]
    for name, (kind, s) in rows.items():
        out.append(
            f"| {name} | {kind} | {s['rmse']:.3f} | {s['bias']:+.3f} | "
            f"{s['M1']:.2f} | {s['M2']:.2f} | {s['M3']:.2f} |"
        )
    return Markdown("\n".join(out))


KIND = {1: "1: empirical, fitted here", 2: "2: pooled C_D, fitted here", 3: "3: other experiment"}
SCORES = {
    "paper's fitted curves (floor)": ("measured", score(paper_fit)),
    "Zhang Eqs. 13–15 (HSVF)": (KIND[1], score(hsvf_model)),
}
closure_scores = {n: score(lambda c, n=n: simulate(c, n)) for n in sd.core.drag.names()}
for n in sorted(closure_scores, key=lambda n: (TIER[n], closure_scores[n]["rmse"])):
    SCORES[f"SADIE + `{n}`"] = (KIND[TIER[n]], closure_scores[n])
display(table(SCORES))

How to read this table:

- **Zhang et al.'s empirical model fits best.** It sits close to the floor, because it
  has one coefficient per model fitted to these same tests (ψ in Eq. 15).
- **With the pooled closures, SADIE predicts the canopy models M2 and M3 to within about
  40 % in β** (mean ratios 0.85–1.41), with the scatter expected of closures fitted to all
  104 tests at once.
- **Every pooled closure overpredicts M1 by a factor of about 3–4.5.** The paper notes
  that its pooled fits "significantly overestimate $C_D$ of cylindrical model M1" (p. 10):
  at a given KC, Ur or $Re_{rv}$, M1 has a much lower bulk $C_D$ than M2 and M3, and a
  single curve fitted through all three cannot represent both.
- **The closures from other experiments split into two groups, and neither predicts all
  three models.** He et al. (whose models include a canopy) and Kelty et al. (prototype
  scale) give large $C_D$: they behave like the pooled closures, with M2 and M3 within
  about 25 % and M1 about 4× too high. The closures fitted to stem-and-root models (Chang,
  Wang, Maza, Bryant) give small $C_D$: they get M1 within a factor of about 1.7 but
  underpredict the canopy models by a factor of 2.5–7.

In [ ]:
names = [
    "Zhang Eqs. 13–15 (HSVF)",
    "SADIE + `zhang2023_kc`",
    "SADIE + `he2019_kc`",
    "SADIE + `chang2022_kc`",
]
models = [
    hsvf_model,
    lambda c: simulate(c, "zhang2023_kc"),
    lambda c: simulate(c, "he2019_kc"),
    lambda c: simulate(c, "chang2022_kc"),
]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4), sharex=True, sharey=True, layout="constrained")
for ax, name, model in zip(axes, names, models, strict=True):
    ax.plot([5e-3, 2], [5e-3, 2], color="0.5", lw=1)
    ax.fill_between([5e-3, 2], [2.5e-3, 1], [1e-2, 4], color="0.93", lw=0)  # factor 2
    for c in CONFIGS:
        pts = [(FIG4_BETA[k[:3]][c], model(k)[1]) for k in CASES if k[3] == c]
        ax.scatter(
            *zip(*pts, strict=True),
            s=40,
            color=COLOR[c],
            marker=MARKER[c],
            edgecolor="white",
            linewidth=1,
            label=c,
            zorder=3,
        )
    ax.set(xscale="log", yscale="log", xlim=(5e-3, 2), ylim=(5e-3, 2), title=name.replace("`", ""))
    ax.set_xlabel("measured β (1/m)")
    ax.title.set_fontsize(9)
axes[0].set_ylabel("predicted β (1/m)")
axes[0].legend(frameon=False)
plt.show()

The grey band is a factor of 2 either side of perfect agreement. The empirical
model sits on the line. With the pooled closure, M2 and M3 fall inside the band and M1
lies above it. He et al.'s closure looks much like the pooled one. Chang et al.'s closure
moves M1 onto the line and drops the canopy models well below it.

## 5. Sensitivity to SADIE's modelling choices

The scores above use Zhang et al.'s analysis convention. Here the pooled closures are rerun
with:

- **resolved geometry and local $C_D$:** SADIE's default, where the canopy width and the
  orbital velocity vary with height and $C_D$ follows the decaying wave (theory §3.9);
- **half the root frontal area:** the inferred root convention of the reproduction notebook
  (§7b), which reproduces the paper's root/stem β ratios and the M1 mean $C_D$.

In [ ]:
rows = {}
for n in ("zhang2023_kc", "zhang2023_ur", "zhang2023_rerv"):
    rows[f"`{n}`, depth-averaged, incident"] = ("as above", closure_scores[n])
    rows[f"`{n}`, resolved, local"] = (
        "SADIE default",
        score(lambda c, n=n: simulate(c, n, "resolved", "local")),
    )
    rows[f"`{n}`, depth-averaged, incident, roots × 0.5"] = (
        "inferred root convention",
        score(lambda c, n=n: simulate(c, n, root_frontal_factor=0.5)),
    )
display(table(rows))

- **Resolved geometry and local $C_D$ damp more** (bias more negative): β for M2 and M3
  rises by 40–60 % with the KC and Ur closures, and doubles with $Re_{rv}$.
  Two effects add up: the resolved canopy dissipates more than its depth average
  (theory §5.3), and $C_D$ rises as the wave decays because every closure falls with KC,
  Ur or Re. The closures were fitted under the averaged convention, so the averaged
  convention is the consistent way to use them.
- **The root convention reduces the M1 error but does not remove it.** With half the root
  area, M1 is still overpredicted by a factor of about 2–3 (1.9 with $Re_{rv}$). The M1
  problem is mainly the pooled closures, not the root geometry. This combination
  (`zhang2023_rerv`, roots × 0.5) gives the lowest RMSE of any physics-based prediction,
  0.066, with M2 and M3 within 10 %; but it uses an inferred convention, so it is reported
  here as a sensitivity, not as SADIE's prediction.

## 6. Summary: how close is SADIE to the flume?

| Prediction | Wave heights (RMSE of H/H0) | Damping factor β |
|---|---|---|
| Paper's own fitted curves | 0.029 | (these define "measured") |
| Zhang et al.'s empirical model (fitted per model to these tests) | 0.039 | within about 10 % on average for each model |
| SADIE + Zhang's pooled $C_D$ closures (same experiment) | 0.09–0.11 | M2, M3: within about 40 %; **M1: 3–4.5× too high** |
| SADIE + closures from other experiments (true predictions) | 0.11–0.19 | either M1 about 4× too high, or M2 and M3 2.5–7× too low |

**Short answer:** the wave physics in SADIE is not the limiting factor. The energy balance
reproduces the paper's own analysis exactly (reproduction notebook) and, given a $C_D$,
the predicted heights decay the way the measured ones do. What decides the agreement is
the drag closure:

- With Zhang et al.'s pooled closures, SADIE gets the canopy models M2 and M3 to within
  about 40 % in β, but cannot represent the stem-and-root model M1.
- No closure from another experiment predicts all three models. This is the scale and
  morphology problem the paper discusses in §4.4: bulk $C_D$ fitted to one plant design
  does not transfer to another.

```{note}
These comparisons use 24 profiles digitized from one figure. The full 104-test dataset
(the paper's data are "available on request") would give a fairer test, especially of
the closures from other experiments. Nothing in this notebook was tuned: the closures,
geometry and conventions are the ones documented in the reproduction notebook.
```